# Load and Merge Reddit Data
Load all CSV files from `Data/raw`, parse metadata from filenames, add labels, and save one merged parquet file in `Data/processed`.

In [4]:
from pathlib import Path
import re
import pandas as pd

# Resolve project paths robustly whether the notebook runs from Notebooks/ or project root.
cwd = Path.cwd().resolve()
if (cwd / "Data" / "raw").exists():
    project_root = cwd
elif (cwd.parent / "Data" / "raw").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not find Data/raw from current working directory.")

raw_dir = project_root / "Data" / "raw"
processed_dir = project_root / "Data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

csv_paths = sorted(raw_dir.glob("*.csv"))
print(f"Found {len(csv_paths)} CSV files in {raw_dir}")

pattern = re.compile(
    r"^(?P<subreddit>.+)_(?P<timeframe>2018|2019|pre|post)_features_tfidf_256\.csv$"
)

mental_health_subreddits = {
    "addiction",
    "adhd",
    "alcoholism",
    "anxiety",
    "autism",
    "bipolarreddit",
    "bpd",
    "depression",
    "edanonymous",
    "healthanxiety",
    "lonely",
    "mentalhealth",
    "ptsd",
    "schizophrenia",
    "socialanxiety",
    "suicidewatch",
}

frames = []
skipped = []

for path in csv_paths:
    match = pattern.match(path.name)
    if not match:
        skipped.append(path.name)
        continue

    subreddit = match.group("subreddit")
    timeframe = match.group("timeframe")

    df = pd.read_csv(path)
    df["subreddit"] = subreddit
    df["timeframe"] = timeframe
    df["is_mental_health"] = int(subreddit.lower() in mental_health_subreddits)
    # Treat only explicit "post" files as COVID-period to avoid labeling 2019 as COVID.
    df["covid_period"] = int(timeframe == "post")

    frames.append(df)

if not frames:
    raise ValueError("No valid CSV files were loaded. Check filename pattern.")

merged_df = pd.concat(frames, ignore_index=True)
output_path = processed_dir / "reddit_mh_merged.parquet"
merged_df.to_parquet(output_path, index=False)

print(f"Loaded files: {len(frames)}")
print(f"Skipped files: {len(skipped)}")
if skipped:
    print("Skipped examples:", skipped[:5])
print(f"Merged shape: {merged_df.shape}")
print(f"Saved merged dataset to: {output_path}")
merged_df.head()

Found 108 CSV files in /Users/ganenthraravindran/Desktop/Mental Health/Data/raw


/var/folders/m_/ptg0zwqn0b34bnnmgnqwtkvm0000gn/T/ipykernel_38828/3389582153.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["timeframe"] = timeframe
/var/folders/m_/ptg0zwqn0b34bnnmgnqwtkvm0000gn/T/ipykernel_38828/3389582153.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["is_mental_health"] = int(subreddit.lower() in mental_health_subreddits)
/var/folders/m_/ptg0zwqn0b34bnnmgnqwtkvm0000gn/T/ipykernel_38828/3389582153.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

Loaded files: 108
Skipped files: 0
Merged shape: (1107302, 353)
Saved merged dataset to: /Users/ganenthraravindran/Desktop/Mental Health/Data/processed/reddit_mh_merged.parquet


,subreddit,author,date,post,automated_readability_index,coleman_liau_index,flesch_kincaid_grade_level,flesch_reading_ease,gulpease_index,gunning_fog_index,...,tfidf_work,tfidf_worri,tfidf_wors,tfidf_would,tfidf_wrong,tfidf_x200b,tfidf_year,timeframe,is_mental_health,covid_period
0,COVID19_support,thatreddittherapist,2020/02/15,"Welcome to COVID19 Support! Hi everyone,\n\nI'...",9.341417,9.238381,8.578618,67.262223,59.609756,11.959475,...,0.059225,0.082616,0.0,0.00000,0.0,0.000000,0.000000,post,0,1
1,COVID19_support,daisy7895528,2020/02/20,Deadman Walking I have Type 2 diabetes and am ...,10.375234,14.460109,7.220383,62.353939,60.869159,11.382804,...,0.000000,0.000000,0.0,0.00000,0.0,0.000000,0.126704,post,0,1
2,COVID19_support,Varakari,2020/02/21,"Constructive Action, Not Pink Glasses There is...",6.023668,9.237107,5.690609,72.257547,71.131980,8.609554,...,0.000000,0.103938,0.0,0.00000,0.0,0.248724,0.000000,post,0,1
3,COVID19_support,ToxicSamurai,2020/02/26,One way I dealt with pandemic panic from SARS-...,9.449293,6.979039,8.541308,75.186932,61.009569,11.202764,...,0.000000,0.000000,0.0,0.09086,0.0,0.000000,0.000000,post,0,1
4,COVID19_support,readyheartsx,2020/02/27,No one around me cares about the COVID19 virus...,-3.395227,-2.115623,-0.876061,111.058636,111.500000,2.181818,...,0.000000,0.000000,0.0,0.00000,0.0,0.000000,0.000000,post,0,1
